In [26]:
def dataset(text, merges):
    train_split = int(len(text) * 0.8)
    test_split= int(len(text) * 0.9)
    
    train_data = text[:train_split]
    test_data = text[train_split:test_split]
    val_data = text[test_split:]

    return train_data, test_data, val_data

In [25]:
def batcher(encoded_ids, block_size: int = 256, batch_size: int = 32):
    X = []
    targets = []

    for i in range(batch_size):
        ix = torch.randint(0, len(encoded_ids) - (block_size + 1), (1, ))

        X.append(encoded_ids[ix:ix+block_size])
        targets.append(encoded_ids[ix+1:ix+block_size+1])

    X = torch.tensor(X)
    targets = torch.tensor(targets)

    return X, targets

In [1]:
def get_stats(ids):

    stats = {}
    
    for pair in zip(ids, ids[1:]):
        if pair in stats:
            stats[pair] += 1
        else:
            stats[pair] = 1

    return stats

In [2]:
def merge(ids, pair, idx):

    new_ids = []

    ix = 0
    while ix < len(ids):
        
        if pair[0] == ids[ix] and ix < len(ids) - 1 and pair[1] == ids[ix+1]:
            new_ids.append(idx)
            ix += 2

        else:
            new_ids.append(ids[ix])
            ix += 1
    return new_ids

In [3]:
def train_bpe(text, num_merge):
    ids = text.encode('utf-8')
    
    merges = {}
    idx = 256
    for i in range(num_merge):
        stats = get_stats(ids)
        if not stats:
            break        
        
        target_pair = max(stats, key=stats.get)
        ids = merge(ids, target_pair, idx)
        merges[target_pair] = idx
        idx += 1

    return merges

In [4]:
def encoder(text, merges):
    encoded_ids = text.encode('utf-8')

    for pair, token_id in merges.items():
        encoded_ids = merge(encoded_ids, pair, token_id)

    return encoded_ids

In [5]:
def decoder(encoded_ids, vocab):
    text = []
    for idx in encoded_ids:
        ch = vocab[idx]
        text.append(ch)

    text = b''.join(text)
    text = text.decode('utf-8')
    
    return text

In [6]:
def create_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for pair, token_id in merges.items():
        vocab[token_id] = vocab[pair[0]] + vocab[pair[1]]

    return vocab

In [7]:
with open("../data/interim/Bianet_tr_minimal-cleaned.txt", mode='r', encoding="utf-8") as file:
    data = file.read()

In [32]:
class BPETokenizer():
    def fit(self, text, num_merge):
        self.text = text
        ids = text.encode('utf-8')
        merges = {}
        idx = 256
        
        for i in range(num_merge):
            stats = get_stats(ids)
            if not stats:
                break        
            
            target_pair = max(stats, key=stats.get)
            ids = merge(ids, target_pair, idx)
            merges[target_pair] = idx
            idx += 1

        vocab = {i: bytes([i]) for i in range(256)}
        for pair, token_id in merges.items():
            vocab[token_id] = vocab[pair[0]] + vocab[pair[1]]
            
        self.merges = merges
        self.vocab = vocab

    def encode(self, text):
        encoded_ids = text.encode('utf-8')
    
        for pair, token_id in self.merges.items():
            encoded_ids = merge(encoded_ids, pair, token_id)

        return encoded_ids

    def decode(self, encoded_ids):
        text = []
        
        for idx in encoded_ids:
            ch = self.vocab[idx]
            text.append(ch)
    
        text = b''.join(text)
        text = text.decode('utf-8')
        
        return text

In [53]:
tokenizer = BPETokenizer()
tokenizer.fit(data[:30000], 50)

In [54]:
sample = data[30000:40000]
encoded_ids = tokenizer.encode(sample)

In [55]:
decoded_text = tokenizer.decode(encoded_ids)

In [56]:
sample == decoded_text

True

In [57]:
len(tokenizer.merges) == 50

True

In [58]:
tokenizer.vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'